# Lecture 03: Reasoning under Uncertainty and Bayesian Networks — Exercises

Companion to the note [Noter små](../Noter%20små.md).

Practise the probability toolkit of Lecture 03 (and the BN inference part of the course): reading a full joint distribution, marginalisation, conditioning, Bayes' rule, (conditional) independence, building Bayesian networks from the chain rule and answering queries by summing out hidden variables, first by hand (exam style) and then in NumPy.

**17 exercises** · 🧠 Concept ×3 · ✍️ By hand ×9 · 💻 Code ×5

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import itertools
# Joint distribution P(A, B): rows a1..a3, columns b1..b3
P_AB = np.array([[0.10, 0.05, 0.05],
                 [0.20, 0.10, 0.05],
                 [0.05, 0.15, 0.25]])
# The three-variable table from the note: P_xyz[x, y, z] with index 0 = x1/y1/z1, 1 = x2/y2/z2
P_xyz = np.array([[[0.10, 0.05], [0.15, 0.10]],
                  [[0.20, 0.05], [0.25, 0.10]]])
# Email counts N[spam, caps, pills] with index 1 = yes, 0 = no
N_email = np.zeros((2, 2, 2))
N_email[1, 1, 1], N_email[1, 1, 0], N_email[1, 0, 1], N_email[1, 0, 0] = 100, 900, 300, 2700
N_email[0, 1, 1], N_email[0, 1, 0], N_email[0, 0, 1], N_email[0, 0, 0] = 6, 294, 114, 5586
# The "exam" Bayesian network  S -> P <- K,  P -> H   (all Boolean; 1 = true)
#   S = studied, K = sick on exam day, P = passed, H = happy
p_S = 0.6                       # P(S = true)
p_K = 0.1                       # P(K = true)
p_P = {(1, 0): 0.9, (1, 1): 0.6, (0, 0): 0.4, (0, 1): 0.1}   # P(P = true | S, K)
p_H = {1: 0.8, 0: 0.3}          # P(H = true | P)

## Part A · Concepts

### Exercise 1 · Causal versus diagnostic knowledge
*🧠 Concept · ★☆☆*

The note says we usually specify $P(\text{toothache}\mid\text{cavity})$ rather than $P(\text{cavity}\mid\text{toothache})$.

1. Which of the two is *causal* and which is *diagnostic*?
2. Suppose a cavity epidemic doubles $P(\text{cavity})$. Which of the two conditional probabilities changes, and which stays the same?
3. Why does this make causal models (and Bayesian networks with arrows from causes to effects) easier to maintain?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about which quantity describes a mechanism (how cavities produce pain) and which depends on how common cavities are.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $P(\text{toothache}\mid\text{cavity})$ is causal (cause → effect); $P(\text{cavity}\mid\text{toothache})$ is diagnostic (effect → cause).
2. The causal probability describes the physiology, so it stays the same. The diagnostic one changes, because by Bayes' rule
   $P(\text{cavity}\mid\text{toothache}) = \frac{P(\text{toothache}\mid\text{cavity})P(\text{cavity})}{P(\text{toothache})}$ depends on the prior $P(\text{cavity})$.
3. With a causal model you only update the prior $P(\text{cavity})$ and Bayes' rule gives every diagnostic probability for free. The
   causal CPTs are also easier for experts to assess and they carry over between populations.

</details>

### Exercise 2 · Query, evidence and hidden variables
*🧠 Concept · ★☆☆*

A Bayesian network has Boolean variables *Cloudy (C)*, *Sprinkler (S)*, *Rain (R)*, *WetGrass (W)* and *Slippery (L)*, with edges
$C\to S$, $C\to R$, $S\to W$, $R\to W$, $W\to L$.

> "This morning it is cloudy and the grass is wet. What is the probability that the sprinkler was on?"

1. Classify every variable as query, evidence or hidden (as in the Sherlock Holmes exam exercise).
2. Write the requested probability, and write it as a ratio of sums over the hidden variables using the BN factorisation.
3. (Extension) $L$ is hidden and has no observed descendants. Show that it can simply be dropped from the sum.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint 1</b></summary>

Query = what is asked for. Evidence = what is observed. Everything else is hidden and must be summed out.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For the extension: $\sum_{l} P(l\mid W) = 1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Query: $S$. Evidence: $C = c$, $W = w$. Hidden: $R$, $L$.
2. $$P(S\mid c, w) = \frac{\sum_{r}\sum_{l} P(c)P(S\mid c)P(r\mid c)P(w\mid S, r)P(l\mid w)}{\sum_{s'}\sum_{r}\sum_{l} P(c)P(s'\mid c)P(r\mid c)P(w\mid s', r)P(l\mid w)}.$$
3. $P(l\mid w)$ is the only factor containing $l$, so it can be pulled to the inside: $\sum_l P(l\mid w) = 1$. Hidden leaves without
   evidence below them ("barren nodes") never influence a query, so you can delete them before computing.

</details>

## Part B · By hand

### Exercise 3 · Marginals and a conditional from a joint table
*✍️ By hand · ★☆☆*

The joint distribution $P(A,B)$ is (rows $a_1,a_2,a_3$; columns $b_1,b_2,b_3$):

|       | $b_1$ | $b_2$ | $b_3$ |
|:-----:|:-----:|:-----:|:-----:|
| $a_1$ | 0.10 | 0.05 | 0.05 |
| $a_2$ | 0.20 | 0.10 | 0.05 |
| $a_3$ | 0.05 | 0.15 | 0.25 |

Compute $P(A)$, $P(B)$ and the distribution $P(A\mid b_2)$. Store them as lists of three numbers.

In [ ]:
P_A = None          # [P(a1), P(a2), P(a3)]
P_B = None          # [P(b1), P(b2), P(b3)]
P_A_given_b2 = None # [P(a1|b2), P(a2|b2), P(a3|b2)]

check('P(A)', P_A, P_AB.sum(axis=1))
check('P(B)', P_B, P_AB.sum(axis=0))
check('P(A | b2)', P_A_given_b2, P_AB[:, 1] / P_AB[:, 1].sum(), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Marginalise: sum each row for $P(A)$ and each column for $P(B)$. For $P(A\mid b_2)$ take the $b_2$ column and divide by its sum.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(A) = (0.20,\ 0.35,\ 0.45)$, $P(B) = (0.35,\ 0.30,\ 0.35)$.

$P(A\mid b_2) = \frac{(0.05,\ 0.10,\ 0.15)}{0.30} = (1/6,\ 1/3,\ 1/2) \approx (0.167,\ 0.333,\ 0.5)$.

```python
P_A = [0.20, 0.35, 0.45]
P_B = [0.35, 0.30, 0.35]
P_A_given_b2 = [1/6, 1/3, 1/2]
```

</details>

### Exercise 4 · Bayes' rule: one test and two tests
*✍️ By hand · ★☆☆*

A disease has prevalence $P(H) = 0.02$. A test has sensitivity $P(E\mid H) = 0.95$ and false positive rate $P(E\mid\neg H) = 0.05$.

1. Compute $P(H\mid E)$ after one positive test.
2. The patient takes a second, independent test (the two results are conditionally independent given the disease status).
   Both are positive. Compute $P(H\mid E_1, E_2)$.

In [ ]:
post_one = None
post_two = None

_l1 = 0.95 * 0.02; check('P(H|E)', post_one, _l1 / (_l1 + 0.05 * 0.98), tol=1e-3)
_l2 = 0.95**2 * 0.02; check('P(H|E1,E2)', post_two, _l2 / (_l2 + 0.05**2 * 0.98), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Denominator: $P(E) = P(E\mid H)P(H) + P(E\mid\neg H)P(\neg H)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Conditional independence gives $P(E_1,E_2\mid H) = P(E_1\mid H)P(E_2\mid H)$. Alternatively use the first posterior as the new prior.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $P(E) = 0.95\cdot0.02 + 0.05\cdot0.98 = 0.019 + 0.049 = 0.068$, so $P(H\mid E) = 0.019/0.068 \approx 0.279$.
2. $P(H\mid E_1,E_2) = \frac{0.95^2\cdot0.02}{0.95^2\cdot0.02 + 0.05^2\cdot0.98} = \frac{0.01805}{0.01805+0.00245} \approx 0.880$.

The low prior keeps the first posterior below 30 %, but a second independent positive pushes it to 88 %. Using the first
posterior $0.279$ as the prior for the second test gives the same number.

```python
post_one = 0.019 / 0.068
post_two = 0.01805 / 0.0205
```

</details>

### Exercise 5 · Size of the CPTs
*✍️ By hand · ★☆☆*

A Bayesian network over six Boolean variables has the parent sets

$\text{Pa}(A)=\emptyset,\ \text{Pa}(B)=\emptyset,\ \text{Pa}(C)=\{A\},\ \text{Pa}(D)=\{A,B\},\ \text{Pa}(E)=\{C,D\},\ \text{Pa}(F)=\{D\}$.

1. Which CPTs must be specified?
2. How many *independent* numbers do they contain in total? (For a Boolean child one number per parent configuration is enough,
   because $P(\neg x\mid\cdot) = 1 - P(x\mid\cdot)$.)
3. How many independent numbers does the full joint distribution over the six variables need?

In [ ]:
n_bn = None
n_joint = None

_pa = {'A': [], 'B': [], 'C': ['A'], 'D': ['A', 'B'], 'E': ['C', 'D'], 'F': ['D']}
check('BN parameters', n_bn, sum(2 ** len(p) for p in _pa.values()))
check('full joint parameters', n_joint, 2 ** len(_pa) - 1)

<details>
<summary><b>💡 Hint</b></summary>

A CPT $P(X\mid \text{Pa}(X))$ with $k$ Boolean parents has $2^k$ rows.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $P(A)$, $P(B)$, $P(C\mid A)$, $P(D\mid A,B)$, $P(E\mid C,D)$, $P(F\mid D)$.
2. $1 + 1 + 2 + 4 + 4 + 2 = 14$ numbers (28 if you also write the complementary entries).
3. $2^6 - 1 = 63$. The BN saves parameters because each variable only depends on its parents.

```python
n_bn = 14
n_joint = 63
```

</details>

### Exercise 6 · Independence from counts
*✍️ By hand · ★★☆*

For 10 000 emails you count the Boolean variables Spam, Caps (subject in capitals) and Pills (body contains "pills"):

|              | Caps, Pills | Caps, ¬Pills | ¬Caps, Pills | ¬Caps, ¬Pills |
|:------------:|:-----------:|:------------:|:------------:|:-------------:|
| **Spam**     | 100 | 900 | 300 | 2700 |
| **¬Spam**    | 6   | 294 | 114 | 5586 |

Decide (True/False):
1. Are Spam and Caps independent?
2. Are Caps and Pills independent?
3. Are Caps and Pills independent **given** Spam?
4. Are Spam and Caps independent given Pills?

In [ ]:
ind_spam_caps = None
ind_caps_pills = None
ind_caps_pills_given_spam = None
ind_spam_caps_given_pills = None

def _indep(T):  # T: 2-D joint table, True if it factorises
    T = T / T.sum(); return bool(np.allclose(T, np.outer(T.sum(1), T.sum(0)), atol=1e-9))
_P = N_email / N_email.sum()
check('Spam _||_ Caps', ind_spam_caps, _indep(_P.sum(2)))
check('Caps _||_ Pills', ind_caps_pills, _indep(_P.sum(0)))
check('Caps _||_ Pills | Spam', ind_caps_pills_given_spam, _indep(_P[0]) and _indep(_P[1]))
check('Spam _||_ Caps | Pills', ind_spam_caps_given_pills, _indep(_P[:, :, 0]) and _indep(_P[:, :, 1]))

<details>
<summary><b>💡 Hint 1</b></summary>

Compare e.g. $P(\text{caps}\mid\text{spam})$ with $P(\text{caps})$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For conditional independence, check $P(\text{caps},\text{pills}\mid\text{spam}) = P(\text{caps}\mid\text{spam})P(\text{pills}\mid\text{spam})$ for **both** values of Spam.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $P(\text{caps}) = 1300/10000 = 0.13$ but $P(\text{caps}\mid\text{spam}) = 1000/4000 = 0.25$: **not independent**.
2. $P(\text{caps},\text{pills}) = 106/10000 = 0.0106$ while $P(\text{caps})P(\text{pills}) = 0.13\cdot0.052 = 0.00676$: **not independent**.
3. Given spam: $P(\text{caps}\mid s)=0.25$, $P(\text{pills}\mid s)=400/4000=0.1$, $P(\text{caps},\text{pills}\mid s)=100/4000=0.025=0.25\cdot0.1$ ✔.
   Given ¬spam: $0.05\cdot0.02 = 0.001 = 6/6000$ ✔. So **independent given Spam**.
4. Given pills: $P(\text{spam}\mid\text{pills}) = 400/520 \approx 0.769$ but $P(\text{spam}\mid\text{caps},\text{pills}) = 100/106 \approx 0.943$: **not independent**.

Lesson: Caps and Pills are dependent only because both are caused by Spam (a common cause). This is exactly the structure
a naive Bayes model assumes (Lecture 07).

```python
ind_spam_caps = False
ind_caps_pills = False
ind_caps_pills_given_spam = True
ind_spam_caps_given_pills = False
```

</details>

### Exercise 7 · Topological orderings and the chain rule
*✍️ By hand · ★☆☆*

A BN has edges $A\to B$, $A\to C$, $C\to D$, $B\to E$, $D\to E$. Which of these orderings are valid topological orderings?

1. $A,B,C,D,E$  2. $A,C,D,B,E$  3. $B,A,C,D,E$  4. $A,C,B,E,D$  5. $C,A,B,D,E$  6. $A,B,C,E,D$

Store the numbers of the valid ones in `valid_orderings`. Then write the chain rule for ordering 2 and simplify it with the graph.

In [ ]:
valid_orderings = None  # e.g. [1, 4]

_edges = [('A','B'), ('A','C'), ('C','D'), ('B','E'), ('D','E')]
_opts = {1: 'ABCDE', 2: 'ACDBE', 3: 'BACDE', 4: 'ACBED', 5: 'CABDE', 6: 'ABCED'}
_ref = [k for k, o in _opts.items() if all(o.index(u) < o.index(v) for u, v in _edges)]
check('valid orderings', sorted(valid_orderings) if valid_orderings is not None else None, _ref)

<details>
<summary><b>💡 Hint</b></summary>

Every parent must come before its children. $E$ has two parents ($B$ and $D$).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Valid: **1 and 2**. Ordering 3 puts $B$ before its parent $A$; 4 and 6 put $E$ before $D$; 5 puts $C$ before $A$.

Chain rule for $A,C,D,B,E$:
$P(A)P(C\mid A)P(D\mid A,C)P(B\mid A,C,D)P(E\mid A,C,D,B)$, which the graph simplifies to
$$P(A)\,P(C\mid A)\,P(D\mid C)\,P(B\mid A)\,P(E\mid B,D).$$

```python
valid_orderings = [1, 2]
```

</details>

### Exercise 8 · Network structure depends on the variable ordering
*🧠 Concept · ★★☆*

Variables (all Boolean): *Mexico* (recently travelled to Mexico), *Infection* (infected with swine flu), *Vaccination*,
*Sick* (sick from swine flu), *Fever*.

1. Build a BN with the ordering Mexico, Infection, Vaccination, Sick, Fever: for each variable, choose as parents the smallest
   subset of the *earlier* variables that makes it conditionally independent of the other earlier ones.
2. Repeat with the ordering Vaccination, Mexico, Sick, Fever, Infection. Count the edges in both networks.
3. What general advice follows?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint 1</b></summary>

Causal story: travelling raises the chance of infection; infection and (no) vaccination together cause sickness; sickness causes fever.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

In the second ordering, Infection comes last, so it needs as parents everything that carries information about it.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Causal ordering: Mexico → Infection; Vaccination has no parents (independent of travel and infection); Infection → Sick ← Vaccination;
   Sick → Fever. **4 edges.**
2. Ordering Vaccination, Mexico, Sick, Fever, Infection: Vaccination and Mexico are roots; Sick depends on both (Mexico → Sick,
   Vaccination → Sick); Fever depends only on Sick; Infection must be explained by Mexico, Sick and Vaccination
   (given that someone is sick, being vaccinated changes the chance they were infected). Edges: V→S, M→S, S→F, M→I, S→I, V→I: **6 edges.**
   Both networks can represent the same distribution, but the second needs more and less intuitive CPTs (e.g. $P(I\mid M,S,V)$).
3. Order the variables **causes before effects**. This gives sparse networks with CPTs that are easy for experts to estimate.

</details>

### Exercise 9 · A full-joint entry from the CPTs
*✍️ By hand · ★☆☆*

The *exam network*: $S$ (studied) and $K$ (sick on exam day) are root nodes, $P$ (passed) has parents $S, K$, and $H$ (happy) has parent $P$.

| $P(s)$ | $P(k)$ |
|:--:|:--:|
| 0.6 | 0.1 |

| $S$ | $K$ | $P(p\mid S,K)$ |
|:--:|:--:|:--:|
| t | f | 0.9 |
| t | t | 0.6 |
| f | f | 0.4 |
| f | t | 0.1 |

| $P$ | $P(h\mid P)$ |
|:--:|:--:|
| t | 0.8 |
| f | 0.3 |

Compute $P(s, \neg k, p, h)$ and $P(\neg s, \neg k, \neg p, h)$. Write which CPT entries you multiply.

In [ ]:
p1 = None  # P(s, ~k, p, h)
p2 = None  # P(~s, ~k, ~p, h)

check('P(s,~k,p,h)', p1, p_S * (1 - p_K) * p_P[(1, 0)] * p_H[1])
check('P(~s,~k,~p,h)', p2, (1 - p_S) * (1 - p_K) * (1 - p_P[(0, 0)]) * p_H[0])

<details>
<summary><b>💡 Hint</b></summary>

$P(S,K,P,H) = P(S)P(K)P(P\mid S,K)P(H\mid P)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(s,\neg k,p,h) = P(s)P(\neg k)P(p\mid s,\neg k)P(h\mid p) = 0.6\cdot0.9\cdot0.9\cdot0.8 = 0.3888$.

$P(\neg s,\neg k,\neg p,h) = 0.4\cdot0.9\cdot(1-0.4)\cdot0.3 = 0.4\cdot0.9\cdot0.6\cdot0.3 = 0.0648$.

```python
p1 = 0.6 * 0.9 * 0.9 * 0.8
p2 = 0.4 * 0.9 * 0.6 * 0.3
```

</details>

### Exercise 10 · Inference by enumeration by hand
*✍️ By hand · ★★☆*

In the exam network, you see that your friend is happy. What is the probability that they studied, $P(s\mid h)$?
Identify query, evidence and hidden variables, then sum out the hidden ones.

In [ ]:
P_s_given_h = None

_J = np.zeros((2, 2, 2, 2))
for _s, _k, _p, _h in itertools.product((0, 1), repeat=4):
    _ps = p_S if _s else 1 - p_S; _pk = p_K if _k else 1 - p_K
    _pp = p_P[(_s, _k)] if _p else 1 - p_P[(_s, _k)]; _ph = p_H[_p] if _h else 1 - p_H[_p]
    _J[_s, _k, _p, _h] = _ps * _pk * _pp * _ph
check('P(s | h)', P_s_given_h, _J[1, :, :, 1].sum() / _J[:, :, :, 1].sum(), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Query $S$, evidence $H=h$, hidden $K$ and $P$. Numerator $\sum_k\sum_p P(s)P(k)P(p\mid s,k)P(h\mid p)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Do it in two steps: first $P(p\mid s) = \sum_k P(k)P(p\mid s,k)$, then $P(h\mid s) = \sum_p P(p\mid s) P(h\mid p)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(p\mid s) = 0.9\cdot0.9 + 0.1\cdot0.6 = 0.87$, so $P(h\mid s) = 0.87\cdot0.8 + 0.13\cdot0.3 = 0.735$ and $P(s,h) = 0.6\cdot0.735 = 0.441$.

$P(p\mid \neg s) = 0.9\cdot0.4 + 0.1\cdot0.1 = 0.37$, so $P(h\mid\neg s) = 0.37\cdot0.8 + 0.63\cdot0.3 = 0.485$ and $P(\neg s,h) = 0.4\cdot0.485 = 0.194$.

$$P(s\mid h) = \frac{0.441}{0.441 + 0.194} = \frac{0.441}{0.635} \approx 0.6945.$$
Seeing the friend happy raises the probability that they studied from 0.6 to about 0.69.

```python
P_s_given_h = 0.441 / 0.635
```

</details>

### Exercise 11 · Coin, die and possible worlds
*✍️ By hand · ★★☆*

Flip a coin. On heads roll a four-sided die, on tails a six-sided die.

1. With a fair coin and fair dice: what is the probability that the number rolled is 3 or 5?
2. Now the coin and dice are loaded, with the distribution over worlds $t_1,\dots,t_6,h_1,\dots,h_4$:
   $t_1 = t_2 = t_4 = t_5 = \tfrac1{12}$, $t_3 = t_6 = \tfrac16$, $h_1 = h_2 = h_3 = h_4 = \tfrac1{12}$.
   Compute $P(\text{tails})$, $P(4\mid\text{tails})$, and $P(\text{roll}\ge4\mid\text{tails})$ and $P(\text{roll}\ge4\mid\text{heads})$.

In [ ]:
p_3or5_fair = None
p_tails = None
p_4_given_tails = None
p_ge4_tails = None
p_ge4_heads = None

_t = np.array([1, 1, 2, 1, 1, 2]) / 12; _h = np.array([1, 1, 1, 1]) / 12
check('P(3 or 5) fair', p_3or5_fair, 0.5 * 1/4 + 0.5 * 2/6)
check('P(tails)', p_tails, _t.sum())
check('P(4 | tails)', p_4_given_tails, _t[3] / _t.sum())
check('P(>=4 | tails)', p_ge4_tails, _t[3:].sum() / _t.sum())
check('P(>=4 | heads)', p_ge4_heads, _h[3:].sum() / _h.sum())

<details>
<summary><b>💡 Hint</b></summary>

Use the set of worlds $S_B$ (coin and die together). Conditioning on tails = keep only the $t_i$ worlds and renormalise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $P(3\text{ or }5) = \tfrac12\cdot\tfrac14 + \tfrac12\cdot\tfrac26 = 0.125 + 0.1\overline{6} \approx 0.2917$ (a four-sided die cannot show 5).
2. $P(\text{tails}) = 4\cdot\tfrac1{12} + 2\cdot\tfrac16 = \tfrac23$. $P(4\mid\text{tails}) = \tfrac{1/12}{2/3} = \tfrac18$.
   $P(\ge4\mid\text{tails}) = \tfrac{1/12+1/12+1/6}{2/3} = \tfrac12$ and $P(\ge4\mid\text{heads}) = \tfrac{1/12}{1/3} = \tfrac14$,
   so the six-sided die is the one more likely to give 4 or more.

```python
p_3or5_fair = 0.125 + 1/6
p_tails = 2/3
p_4_given_tails = 1/8
p_ge4_tails = 0.5
p_ge4_heads = 0.25
```

</details>

### Exercise 12 · Why naive Bayes scales
*✍️ By hand · ★★☆*

A spam classifier uses $n$ Boolean word features $a_1,\dots,a_n$ and the class Spam. Count the independent parameters for

1. the full joint distribution $P(a_1,\dots,a_n,\text{Spam})$;
2. the naive Bayes network $P(\text{Spam})\prod_i P(a_i\mid\text{Spam})$ from Lecture 07.

Give both numbers for $n = 30$.

In [ ]:
n_full_30 = None
n_nb_30 = None

check('full joint', n_full_30, 2 ** 31 - 1)
check('naive Bayes', n_nb_30, 1 + 2 * 30)

<details>
<summary><b>💡 Hint</b></summary>

The joint has $2^{n+1}$ cells that sum to 1. Each $P(a_i\mid\text{Spam})$ needs one number per value of Spam.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $2^{n+1} - 1 = 2^{31} - 1 = 2\,147\,483\,647$.
2. $1 + 2n = 61$. Linear instead of exponential growth: that is why the strong independence assumption is attractive, even when it is not exactly true.

```python
n_full_30 = 2 ** 31 - 1
n_nb_30 = 61
```

</details>

## Part C · Code

### Exercise 13 · Marginalise and condition a joint table
*💻 Code · ★☆☆*

Joint distributions are just NumPy arrays with one axis per variable. Implement

* `marginalize(P, axes)`: sums out the given axes (tuple of ints),
* `conditional(P, query_axis, evidence)`: returns the distribution over `query_axis` given `evidence`, a dict `{axis: value}`;
  all other axes are hidden and summed out.

Test it on the note's table `P_xyz[x, y, z]`: compute $P(Z)$ and $P(X\mid z_1)$.

In [ ]:
def marginalize(P, axes):
    # TODO
    return None

def conditional(P, query_axis, evidence):
    # TODO: fix the evidence values, sum out hidden axes, normalise
    return None

P_Z = marginalize(P_xyz, (0, 1))
P_X_given_z1 = conditional(P_xyz, 0, {2: 0})

check('P(Z)', P_Z, [0.70, 0.30])
check('P(X | z1)', P_X_given_z1, [0.25 / 0.70, 0.45 / 0.70])

<details>
<summary><b>💡 Hint 1</b></summary>

`P.sum(axis=axes)` works with a tuple of axes.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For conditioning, build an index tuple with `slice(None)` everywhere except the evidence axes, then sum out every remaining axis except the query axis.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(Z) = (0.70, 0.30)$ and $P(X\mid z_1) = (0.25, 0.45)/0.70 \approx (0.357, 0.643)$, as in the note. Note that after indexing the
evidence axes away, the query axis may have a new position, so it is easiest to keep the evidence axes with a length-1 slice.

```python
def marginalize(P, axes):
    return P.sum(axis=tuple(axes))

def conditional(P, query_axis, evidence):
    idx = [slice(None)] * P.ndim
    for ax, val in evidence.items():
        idx[ax] = slice(val, val + 1)          # keep the axis (length 1)
    sub = P[tuple(idx)]
    others = tuple(a for a in range(P.ndim) if a != query_axis)
    unnorm = sub.sum(axis=others)
    return unnorm / unnorm.sum()

P_Z = marginalize(P_xyz, (0, 1))
P_X_given_z1 = conditional(P_xyz, 0, {2: 0})
```

</details>

### Exercise 14 · Build the full joint from the CPTs
*💻 Code · ★★☆*

Write `build_joint()` returning a `(2, 2, 2, 2)` array `J[s, k, p, h]` for the exam network, using `p_S`, `p_K`, `p_P`, `p_H`
from the setup cell (index 1 = true). Verify that it sums to 1 and that it reproduces your answers to the two by-hand entries.

In [ ]:
def build_joint():
    # TODO: loop over all 16 assignments (itertools.product) and multiply the CPT entries
    return None

J = build_joint()

check('sums to one', None if J is None else J.sum(), 1.0)
check('J[s,~k,p,h]', None if J is None else J[1, 0, 1, 1], 0.6 * 0.9 * 0.9 * 0.8)

<details>
<summary><b>💡 Hint</b></summary>

Write a small helper `bern(p, x)` that returns `p if x else 1 - p`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The joint has $2^4 = 16$ entries but is fully determined by the $1+1+4+2 = 8$ CPT numbers.

```python
def build_joint():
    bern = lambda p, x: p if x else 1 - p
    J = np.zeros((2, 2, 2, 2))
    for s, k, p, h in itertools.product((0, 1), repeat=4):
        J[s, k, p, h] = bern(p_S, s) * bern(p_K, k) * bern(p_P[(s, k)], p) * bern(p_H[p], h)
    return J

J = build_joint()
```

</details>

### Exercise 15 · Explaining away
*💻 Code · ★★☆*

Use your `conditional` function on `J` (axes: 0 = S, 1 = K, 2 = P, 3 = H) to compute

* `pk_prior` $= P(k)$,
* `pk_fail` $= P(k\mid\neg p)$,
* `pk_fail_studied` $= P(k\mid\neg p, s)$,
* `pk_fail_lazy` $= P(k\mid\neg p, \neg s)$.

Explain the pattern. Are $S$ and $K$ independent? Are they independent given $P$?

In [ ]:
pk_prior = None
pk_fail = None
pk_fail_studied = None
pk_fail_lazy = None

_JJ = np.zeros((2, 2, 2, 2))
for _s, _k, _p, _h in itertools.product((0, 1), repeat=4):
    _JJ[_s, _k, _p, _h] = ((p_S if _s else 1 - p_S) * (p_K if _k else 1 - p_K)
                           * (p_P[(_s, _k)] if _p else 1 - p_P[(_s, _k)]) * (p_H[_p] if _h else 1 - p_H[_p]))
check('P(k)', pk_prior, 0.1)
check('P(k | ~p)', pk_fail, _JJ[:, 1, 0, :].sum() / _JJ[:, :, 0, :].sum())
check('P(k | ~p, s)', pk_fail_studied, _JJ[1, 1, 0, :].sum() / _JJ[1, :, 0, :].sum())
check('P(k | ~p, ~s)', pk_fail_lazy, _JJ[0, 1, 0, :].sum() / _JJ[0, :, 0, :].sum())

<details>
<summary><b>💡 Hint</b></summary>

`conditional(J, 1, {2: 0, 0: 1})[1]` is $P(k\mid\neg p, s)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(k) = 0.1$, $P(k\mid\neg p) \approx 0.182$, $P(k\mid\neg p,s) \approx 0.308$, $P(k\mid\neg p,\neg s) \approx 0.143$.

Failing makes sickness more likely (0.1 → 0.18). If we also learn the student studied, the failure needs another explanation, so
sickness becomes even more likely (0.31). If they did not study, the laziness *explains away* the failure and sickness drops back (0.14).
$S$ and $K$ are independent a priori (two root nodes) but become **dependent given their common effect** $P$ (a v-structure).

```python
pk_prior = conditional(J, 1, {})[1]
pk_fail = conditional(J, 1, {2: 0})[1]
pk_fail_studied = conditional(J, 1, {2: 0, 0: 1})[1]
pk_fail_lazy = conditional(J, 1, {2: 0, 0: 0})[1]
print(pk_prior, pk_fail, pk_fail_studied, pk_fail_lazy)
```

</details>

### Exercise 16 · Enumeration without the full joint
*💻 Code · ★★☆*

Building the full joint is exponential in the number of variables. Write `enumerate_s_given_h(h)` that computes $P(S\mid H=h)$
directly from the CPTs by summing over the hidden variables $K$ and $P$, i.e. by evaluating
$$P(S\mid h) \propto \sum_k\sum_p P(S)P(k)P(p\mid S,k)P(h\mid p),$$
and returns `[P(~s|h), P(s|h)]`. Compare with your by-hand answer.

In [ ]:
def enumerate_s_given_h(h):
    # TODO
    return None

post_h = enumerate_s_given_h(1)
post_not_h = enumerate_s_given_h(0)

_Jr = (np.array([1 - p_S, p_S])[:, None, None, None] * np.array([1 - p_K, p_K])[None, :, None, None]
       * np.array([[[1 - p_P[(s, k)], p_P[(s, k)]] for k in (0, 1)] for s in (0, 1)])[:, :, :, None]
       * np.array([[1 - p_H[0], p_H[0]], [1 - p_H[1], p_H[1]]])[None, None, :, :])
check('P(S | h)', post_h, _Jr[:, :, :, 1].sum((1, 2)) / _Jr[:, :, :, 1].sum())
check('P(S | ~h)', post_not_h, _Jr[:, :, :, 0].sum((1, 2)) / _Jr[:, :, :, 0].sum())

<details>
<summary><b>💡 Hint</b></summary>

Two nested loops over `k` and `p` for each value of `s`; normalise at the end (the denominator is $P(h)$).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(s\mid h) \approx 0.6945$ (as by hand) and $P(s\mid\neg h) \approx 0.436$. Computing unnormalised values for each query value
and normalising at the end avoids computing $P(h)$ separately. Variable elimination improves on this by caching the inner sums.

```python
def enumerate_s_given_h(h):
    bern = lambda p, x: p if x else 1 - p
    un = np.zeros(2)
    for s in (0, 1):
        for k in (0, 1):
            for p in (0, 1):
                un[s] += bern(p_S, s) * bern(p_K, k) * bern(p_P[(s, k)], p) * bern(p_H[p], h)
    return un / un.sum()

post_h = enumerate_s_given_h(1)
post_not_h = enumerate_s_given_h(0)
print(post_h, post_not_h)
```

</details>

### Exercise 17 · Approximate inference by rejection sampling
*💻 Code · ★★★*

Estimate $P(s\mid h)$ by **rejection sampling**: draw `n = 50_000` samples from the network in topological order
($S$, $K$, then $P$, then $H$), throw away samples with $H$ false, and report the fraction of the rest with $S$ true.
Use `rng = np.random.default_rng(0)` and vectorised NumPy. How many samples were kept?

In [ ]:
rng = np.random.default_rng(0)
n = 50_000
est_s_given_h = None
n_kept = None

check('estimate within 0.01 of exact', None if est_s_given_h is None else abs(est_s_given_h - 0.441 / 0.635) < 0.01, True)

<details>
<summary><b>💡 Hint</b></summary>

`S = rng.random(n) < p_S`. For $P$ build the per-sample probability with `np.where`, e.g. `np.where(S & ~K, 0.9, ...)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Roughly $0.635\cdot n \approx 31\,750$ samples survive, and the estimate is within about $\pm0.005$ of $0.6945$
(standard error $\sqrt{0.69\cdot0.31/31750}\approx0.0026$). Rejection sampling wastes samples when the evidence is unlikely,
which motivates likelihood weighting.

```python
rng = np.random.default_rng(0)
n = 50_000
S = rng.random(n) < p_S
K = rng.random(n) < p_K
pp = np.where(S & ~K, p_P[(1, 0)], np.where(S & K, p_P[(1, 1)], np.where(~S & ~K, p_P[(0, 0)], p_P[(0, 1)])))
P = rng.random(n) < pp
H = rng.random(n) < np.where(P, p_H[1], p_H[0])
n_kept = int(H.sum())
est_s_given_h = S[H].mean()
print(n_kept, est_s_given_h)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Noter små](../Noter%20små.md).*